In [11]:
import math 
import os 
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd 
from scipy.special import gamma
import pydmd

In [3]:
savepath = "C:\\Users\\natal\\Thesis"
if not os.path.exists(savepath):
    os.makedirs(savepath) 


os.chdir(savepath)

In [4]:
#load in csv files of minutely closing prices of crypto currencies 
df_cryp = pd.read_csv('crypto_data.csv')
df_cryp.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
df_cryp['Date'] = pd.to_datetime(df_cryp['Date'])

In [5]:
df_cryp.head()

,Date,Time_int,Close_BTCUSD,Close_ETHUSD,Close_USDTUSD,Close_XRPUSD,Close_ADAUSD
0,2019-12-31 17:05:00,1577837100000,7154.60,128.930,0.9962,0.193020,0.032574
1,2019-12-31 17:09:00,1577837340000,7161.70,128.360,0.9962,0.192410,0.032760
2,2019-12-31 17:10:00,1577837400000,7202.10,128.845,0.9963,0.193380,0.032771
3,2019-12-31 17:11:00,1577837460000,7189.63,128.240,0.9967,0.192948,0.032764
4,2019-12-31 17:17:00,1577837820000,7156.93,128.420,0.9967,0.192919,0.032782


In [6]:
#calculate annualized realized volatilities

In [7]:
vars_dict = {
    'Close_BTCUSD': [],
    'Close_ETHUSD': [],
    'Close_USDTUSD': [],
    'Close_XRPUSD': [],
    'Close_ADAUSD': []
}

for hour, group in df_cryp.groupby(pd.Grouper(key='Date', freq='h')):
    for name in vars_dict.keys():
        closes = group[name].values

        rets = np.log(closes[1:] / closes[:-1])

        hourly_rv = 100 * np.sqrt(252/60 * np.sum(rets**2))

        vars_dict[name].append((hour, hourly_rv))

In [8]:
vars_eurusd = vars_dict['Close_BTCUSD']
vars_gbpusd = vars_dict['Close_ETHUSD']
vars_jpyusd = vars_dict['Close_USDTUSD']
vars_audusd = vars_dict['Close_XRPUSD']
vars_cadusd = vars_dict['Close_ADAUSD']
df_vars = pd.DataFrame({
    'Date': [x[0] for x in vars_eurusd],
    'RV_BTCUSD': [x[1] for x in vars_eurusd],
    'RV_ETHUSD': [x[1] for x in vars_gbpusd],
    'RV_USDTUSD': [x[1] for x in vars_jpyusd],
    'RV_XRPUSD': [x[1] for x in vars_audusd],
    'RV_ADAUSD': [x[1] for x in vars_cadusd]})

#remove first row with 0 values
df_vars = df_vars.iloc[1:].reset_index(drop=True)

In [9]:
# need to remove rows with 0 values before proceeding further
df_vars = df_vars[(df_vars.T != 0).all()].reset_index(drop=True)

In [10]:
df_vars.head()

,Date,RV_BTCUSD,RV_ETHUSD,RV_USDTUSD,RV_XRPUSD,RV_ADAUSD
0,2019-12-31 18:00:00,2.807478,2.618315,0.163277,2.518782,1.761112
1,2019-12-31 19:00:00,0.953811,1.594429,0.207727,1.826199,0.546623
2,2019-12-31 20:00:00,2.307927,3.012159,0.161921,3.264336,0.588951
3,2019-12-31 21:00:00,2.222897,2.517143,0.175708,2.840797,1.493980
4,2019-12-31 22:00:00,0.793911,0.839264,0.116362,0.556730,0.190038
